# Token-Level Fovea LoRA Training

VQ-VAE always sees **clean pixels** → zero image-level OOD.  
After VQ encoding, a circular bg_token mask is applied to peripheral tokens.  
LoRA teaches the model: `bg_token in periphery = don't attend`.

**Bridge data path:** `/content/drive/MyDrive/univla_foveated/bridge_data_v2/0.0.1`

In [ ]:
# ── Cell 1: Mount Drive & set paths ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess

# Bridge TFDS data (already on Drive)
BRIDGE_DIR   = "/content/drive/MyDrive/univla_foveated/bridge_data_v2/0.0.1"

# UniVLA pretrained weights
EMU_HUB      = "/content/pretrain/UNIVLA_SIMPLER_BRIDGE_VIDEO_BS128_20K"
VISION_HUB   = "/content/pretrain/Emu3-VisionTokenizer"
FAST_PATH    = "/content/UniVLA/pretrain"

# Output paths
CLEAN_DIR    = "/content/bridge_clean"        # clean frames + fovea centers
PICKLE_PATH  = "/content/bridge_token_fovea_train.pkl"
LORA_DIR     = "/content/lora_token_fovea"    # LoRA adapter output

# Verify Bridge data exists
assert os.path.exists(BRIDGE_DIR), f"Bridge data not found: {BRIDGE_DIR}"
print(f"[OK] Bridge TFDS found: {BRIDGE_DIR}")
!ls "{BRIDGE_DIR}" | head -5

In [ ]:
# ── Cell 2: Extract clean frames + DINO fovea centers ─────────────────────────
# Runtime: ~45-90 min for 500 episodes on Colab GPU
# Adjust --max-episodes for faster testing (e.g. 50 for a quick sanity check)

!conda run -n univla python \
    /content/UniVLA/experiments/foveated_tokenization/lora_training/02_process_clean.py \
    --dataset-dir  "{BRIDGE_DIR}" \
    --output-dir   "{CLEAN_DIR}" \
    --max-episodes 500 \
    --device       cuda

print("\n[done] Clean frames saved to:", CLEAN_DIR)
!ls "{CLEAN_DIR}" | wc -l

In [ ]:
# ── Cell 3: Build training pickle ─────────────────────────────────────────────

!conda run -n univla python \
    /content/UniVLA/experiments/foveated_tokenization/lora_training/03_make_pickle_token_fovea.py \
    --processed-dir "{CLEAN_DIR}" \
    --output-pkl    "{PICKLE_PATH}" \
    --min-frames    8

import pickle
with open(PICKLE_PATH, "rb") as f:
    data = pickle.load(f)
n_fovea = sum(1 for e in data if 'fovea_center' in e)
print(f"\n[OK] Pickle: {len(data)} episodes, {n_fovea} with DINO fovea center")
print("Example fovea_center:", data[0].get('fovea_center', 'N/A'))

In [ ]:
# ── Cell 4: LoRA fine-tuning ──────────────────────────────────────────────────
# fovea_fraction=0.4 → radius = 32*0.4 ≈ 12.8 tokens (same as zero-shot TokenFovea)
# Estimated: ~2-3h for 3 epochs × 500 episodes on A100

!conda run -n univla python \
    /content/UniVLA/experiments/foveated_tokenization/lora_training/train_lora_token_fovea.py \
    --emu-hub        "{EMU_HUB}" \
    --vision-hub     "{VISION_HUB}" \
    --fast-path      "{FAST_PATH}" \
    --data-pkl       "{PICKLE_PATH}" \
    --output-dir     "{LORA_DIR}" \
    --num-epochs     3 \
    --batch-size     1 \
    --grad-accum     8 \
    --lr             1e-4 \
    --fovea-fraction 0.4

print("\n[done] LoRA adapter saved to:", LORA_DIR + "/lora_adapter")
!ls "{LORA_DIR}/lora_adapter"

In [ ]:
# ── Cell 5: Evaluate LoRA adapter on SimplerEnv ───────────────────────────────
# Compare: baseline vs token_fovea (zero-shot) vs token_fovea_lora (adapted)

LORA_ADAPTER = LORA_DIR + "/lora_adapter"

!conda run -n univla python \
    /content/UniVLA/experiments/foveated_tokenization/run_eval_compare.py \
    --emu-hub      "{EMU_HUB}" \
    --vision-hub   "{VISION_HUB}" \
    --fast-path    "{FAST_PATH}" \
    --env-name     widowx_stack_cube \
    --num-episodes 20 \
    --max-steps    200 \
    --lora-path    "{LORA_ADAPTER}" \
    --token-fovea-only \
    --fovea-fraction 0.4 \
    --output-dir   /content/eval_lora_results \
    --save-video